# Pong básico con `turtle` — tu primer videojuego

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maximofernandezriera/pong-turtle/blob/main/pong-turtle.ipynb)

Cuaderno de lectura y construcción guiada del **Pong** con la librería estándar `turtle` de Python.

> **Importante:** `turtle` abre una ventana real (tkinter). **No funciona dentro de Colab ni de Jupyter en el navegador.** Este cuaderno sirve para entender y montar el juego paso a paso; para jugarlo hay que copiar el código a tu equipo y ejecutarlo allí (IDLE, Thonny, VS Code o terminal). Al final está el `pong.py` completo.

## 1 · ¿Qué es `turtle`?

`turtle` viene instalada con Python. Es una tortuga que dibuja moviéndose por un lienzo con **coordenadas cartesianas**:

```
        y
        ↑
        |        (0, 300)
(-400,0)┼──(0,0)──→ x
        |
        |        (0, -300)
```

- El **centro** de la ventana es `(0, 0)`. Arriba y a la derecha es positivo.
- Un `Turtle` tiene posición (`xcor()`, `ycor()`), forma y color. Nuestras palas y la pelota **son tortugas**.
- `pantalla.tracer(0)` apaga el dibujo automático y `pantalla.update()` repinta cada fotograma: así se hace animación fluida.

## 2 · La ventana y el marcador

In [ ]:
import turtle

ANCHO, ALTO = 800, 600

pantalla = turtle.Screen()
pantalla.title("Pong básico")
pantalla.bgcolor("black")
pantalla.setup(width=ANCHO, height=ALTO)
pantalla.tracer(0)          # dibujamos a mano para que no parpadee

puntos_izq = 0
puntos_der = 0

marcador = turtle.Turtle()
marcador.hideturtle()
marcador.penup()
marcador.color("white")
marcador.goto(0, ALTO // 2 - 50)

def escribir_marcador():
    marcador.clear()
    marcador.write(f"{puntos_izq}   :   {puntos_der}",
                   align="center", font=("Courier", 24, "bold"))

## 3 · Las palas

Las dos palas son iguales: un cuadrado (`"square"`) **estirado** en vertical. Por eso usamos una función `crear_pala(x)` — crear la izquierda y la derecha es la misma operación con distinto argumento.

In [ ]:
def crear_pala(x):
    pala = turtle.Turtle()
    pala.shape("square")
    pala.color("white")
    pala.shapesize(stretch_wid=5, stretch_len=1)  # 100x20 px
    pala.penup()
    pala.goto(x, 0)
    return pala

pala_izq = crear_pala(-ANCHO // 2 + 40)   # x = -360
pala_der = crear_pala(ANCHO // 2 - 40)    # x =  360

## 4 · La pelota y su velocidad

La pelota lleva dos atributos propios: `dx` y `dy`, los **píxeles que avanza en cada fotograma**. Son su velocidad en cada eje.

> **Idea clave:** *rebotar* no es más que **cambiar el signo** de `dx` o `dy`.

In [ ]:
pelota = turtle.Turtle()
pelota.shape("square")
pelota.color("white")
pelota.penup()
pelota.goto(0, 0)
pelota.dx = 0.25
pelota.dy = 0.25

def reiniciar_pelota():
    pelota.goto(0, 0)
    pelota.dx = -pelota.dx   # saca el jugador que acaba de recibir el punto

## 5 · Los controles

`turtle` registra funciones para teclas con `pantalla.onkey(función, "tecla")`. Cada función solo mueve la pala **si no se sale de la pantalla** (un `if` de rango).

In [ ]:
def izq_arriba():
    if pala_izq.ycor() < ALTO // 2 - 60:
        pala_izq.sety(pala_izq.ycor() + 20)

def izq_abajo():
    if pala_izq.ycor() > -ALTO // 2 + 60:
        pala_izq.sety(pala_izq.ycor() - 20)

def der_arriba():
    if pala_der.ycor() < ALTO // 2 - 60:
        pala_der.sety(pala_der.ycor() + 20)

def der_abajo():
    if pala_der.ycor() > -ALTO // 2 + 60:
        pala_der.sety(pala_der.ycor() - 20)

pantalla.listen()                    # la ventana empieza a escuchar el teclado
pantalla.onkey(izq_arriba, "w")
pantalla.onkey(izq_abajo, "s")
pantalla.onkey(der_arriba, "Up")
pantalla.onkey(der_abajo, "Down")

## 6 · El bucle principal — el corazón del juego

Todo videojuego tiene un **game loop**: un `while` que, en cada vuelta,

1. repinta la pantalla,
2. mueve lo que se mueve solo (la pelota),
3. comprueba los `if` del mundo: rebotes, goles y colisiones.

In [ ]:
escribir_marcador()

while True:
    pantalla.update()

    # mover la pelota
    pelota.setx(pelota.xcor() + pelota.dx)
    pelota.sety(pelota.ycor() + pelota.dy)

    # rebote arriba / abajo
    if pelota.ycor() > ALTO // 2 - 10 or pelota.ycor() < -ALTO // 2 + 10:
        pelota.dy = -pelota.dy

    # punto para la derecha (la pelota sale por la izquierda)
    if pelota.xcor() < -ANCHO // 2:
        puntos_der += 1
        escribir_marcador()
        reiniciar_pelota()

    # punto para la izquierda (la pelota sale por la derecha)
    if pelota.xcor() > ANCHO // 2:
        puntos_izq += 1
        escribir_marcador()
        reiniciar_pelota()

    # rebote en las palas: ¿está cerca en x y a su altura en y?
    if pelota.xcor() > ANCHO // 2 - 60 and pelota.dx > 0:
        if abs(pelota.ycor() - pala_der.ycor()) < 50:
            pelota.dx = -pelota.dx

    if pelota.xcor() < -ANCHO // 2 + 60 and pelota.dx < 0:
        if abs(pelota.ycor() - pala_izq.ycor()) < 50:
            pelota.dx = -pelota.dx

## 7 · Cómo ejecutarlo

`turtle` necesita una pantalla, así que el juego **se ejecuta en tu equipo**, no en Colab:

- **IDLE** (viene con Python): crea `pong.py`, pega el código completo de abajo y Run (F5).
- **Thonny**: igual, pegar y ejecutar.
- **Terminal**: `python3 pong.py` (en Windows, `python pong.py`).

Jugador izquierdo: **W / S** · Jugador derecho: **flechas ↑ / ↓**. Cierra la ventana para parar.

## 8 · `pong.py` completo

Copia esta celda entera a un fichero `pong.py` en tu equipo:

In [ ]:
"""Pong mínimo con turtle — versión para clase (DAW / Think Python).

Dos jugadores:
  - Izquierdo: teclas W (subir) y S (bajar)
  - Derecho:   flechas Arriba / Abajo

Conceptos que practica: bucle principal, coordenadas, condicionales,
colisiones y funciones.
"""
import turtle

ANCHO, ALTO = 800, 600

# ---------- ventana ----------
pantalla = turtle.Screen()
pantalla.title("Pong básico")
pantalla.bgcolor("black")
pantalla.setup(width=ANCHO, height=ALTO)
pantalla.tracer(0)          # dibujamos a mano para que no parpadee

# ---------- marcador ----------
puntos_izq = 0
puntos_der = 0

marcador = turtle.Turtle()
marcador.hideturtle()
marcador.penup()
marcador.color("white")
marcador.goto(0, ALTO // 2 - 50)

def escribir_marcador():
    marcador.clear()
    marcador.write(f"{puntos_izq}   :   {puntos_der}",
                   align="center", font=("Courier", 24, "bold"))

# ---------- palas ----------
def crear_pala(x):
    pala = turtle.Turtle()
    pala.shape("square")
    pala.color("white")
    pala.shapesize(stretch_wid=5, stretch_len=1)  # 100x20 px
    pala.penup()
    pala.goto(x, 0)
    return pala

pala_izq = crear_pala(-ANCHO // 2 + 40)
pala_der = crear_pala(ANCHO // 2 - 40)

# ---------- pelota ----------
pelota = turtle.Turtle()
pelota.shape("square")
pelota.color("white")
pelota.penup()
pelota.goto(0, 0)
pelota.dx = 0.25          # velocidad: píxeles por fotograma
pelota.dy = 0.25

def reiniciar_pelota():
    pelota.goto(0, 0)
    pelota.dx = -pelota.dx   # saca el que acaba de recibir el punto

# ---------- controles ----------
def izq_arriba():
    if pala_izq.ycor() < ALTO // 2 - 60:
        pala_izq.sety(pala_izq.ycor() + 20)

def izq_abajo():
    if pala_izq.ycor() > -ALTO // 2 + 60:
        pala_izq.sety(pala_izq.ycor() - 20)

def der_arriba():
    if pala_der.ycor() < ALTO // 2 - 60:
        pala_der.sety(pala_der.ycor() + 20)

def der_abajo():
    if pala_der.ycor() > -ALTO // 2 + 60:
        pala_der.sety(pala_der.ycor() - 20)

pantalla.listen()
pantalla.onkey(izq_arriba, "w")
pantalla.onkey(izq_abajo, "s")
pantalla.onkey(der_arriba, "Up")
pantalla.onkey(der_abajo, "Down")

escribir_marcador()

# ---------- bucle principal ----------
while True:
    pantalla.update()

    # mover la pelota
    pelota.setx(pelota.xcor() + pelota.dx)
    pelota.sety(pelota.ycor() + pelota.dy)

    # rebote arriba / abajo
    if pelota.ycor() > ALTO // 2 - 10 or pelota.ycor() < -ALTO // 2 + 10:
        pelota.dy = -pelota.dy

    # punto para la derecha (la pelota sale por la izquierda)
    if pelota.xcor() < -ANCHO // 2:
        puntos_der += 1
        escribir_marcador()
        reiniciar_pelota()

    # punto para la izquierda (la pelota sale por la derecha)
    if pelota.xcor() > ANCHO // 2:
        puntos_izq += 1
        escribir_marcador()
        reiniciar_pelota()

    # rebote en las palas: está a su altura x y dentro de su rango y
    if pelota.xcor() > ANCHO // 2 - 60 and pelota.dx > 0:
        if abs(pelota.ycor() - pala_der.ycor()) < 50:
            pelota.dx = -pelota.dx

    if pelota.xcor() < -ANCHO // 2 + 60 and pelota.dx < 0:
        if abs(pelota.ycor() - pala_izq.ycor()) < 50:
            pelota.dx = -pelota.dx

## 9 · Retos para casa

1. **Acelera el juego**: sube `pelota.dx` / `pelota.dy` a `0.4`. ¿Qué notas?
2. **La pelota acelera**: cada vez que rebota en una pala, multiplica `pelota.dx` por `1.1` (con tope).
3. **Gana quien llegue a 11**: convierte el `while True` en `while puntos_izq < 11 and puntos_der < 11` y muestra "¡Gana el jugador X!" al final.
4. **Cambia el saque**: en `reiniciar_pelota`, haz que la pelota salga con un `dy` aleatorio (`import random`).
5. **Línea central**: dibuja una línea discontinua vertical con otra tortuga que salte `penup`/`pendown`.

Y si te atascas: escribe primero el plan en castellano (qué recibo → qué hago → qué devuelvo), como en la hoja de ruta de los katas.